In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import TweedieRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from scipy.stats import spearmanr

In [2]:
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
df_train = pd.read_csv("data/train_final.csv")

In [4]:
df_train.head()

,cust_id,sale_id_nunique,prod_id_count,sale_revenue_sum,sale_revenue_mean,sale_discount_applied_sum,is_returned_mean,is_returned_last,pack_delay_days_max,is_jan_july_mean,prod_web_only_mean,prod_brand_mode,prod_size_nunique,recency_days,discount_affinity,recent_spend_ratio,spend_trajectory_ratio,churn_risk_ratio,revenue_2018_2019
0,222agnowc53dykbq,1,1,89.95,89.950000,0.00,0.000000,0,0,0.0,0.0,geox,1,383,0.000000,0.0,0.0,383.0,0.0
1,222ny4m63rmalpdl,1,3,125.93,41.976667,-45.16,0.333333,0,0,0.0,0.0,other,3,374,-0.358612,0.0,0.0,37400.0,0.0
2,223jend5smd4ptmc,1,1,89.95,89.950000,0.00,0.000000,0,0,0.0,0.0,tommy hilfiger,1,488,0.000000,0.0,0.0,488.0,0.0
3,223xvc4rbjatlnev,1,2,116.14,58.070000,-49.76,0.000000,0,7,1.0,0.0,teva,2,520,-0.428448,0.0,0.0,52000.0,0.0
4,223y2r357elerbis,1,2,47.97,23.985000,-58.96,0.500000,0,0,1.0,0.0,tamaris,1,348,-1.229102,0.0,479700.0,34800.0,0.0


In [5]:
df_train.columns

Index(['cust_id', 'sale_id_nunique', 'prod_id_count', 'sale_revenue_sum',
       'sale_revenue_mean', 'sale_discount_applied_sum', 'is_returned_mean',
       'is_returned_last', 'pack_delay_days_max', 'is_jan_july_mean',
       'prod_web_only_mean', 'prod_brand_mode', 'prod_size_nunique',
       'recency_days', 'discount_affinity', 'recent_spend_ratio',
       'spend_trajectory_ratio', 'churn_risk_ratio', 'revenue_2018_2019'],
      dtype='str')

In [6]:
TARGET_COLUMN = "revenue_2018_2019"

In [7]:
X_train = df_train.drop(columns=["cust_id", TARGET_COLUMN])

In [8]:
y_train = df_train[TARGET_COLUMN]

In [9]:
df_valid = pd.read_csv("data/valid_final.csv")

In [10]:
X_valid = df_valid.drop(columns=["cust_id", TARGET_COLUMN])

In [11]:
y_valid = df_valid[TARGET_COLUMN]

In [12]:
categorical_cols = X_train.select_dtypes(include=['str', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

In [13]:
numeric_cols = [col for col in numeric_cols if col != 'cust_id']

In [14]:
categorical_cols

['prod_brand_mode']

In [15]:
numeric_cols

['sale_id_nunique',
 'prod_id_count',
 'sale_revenue_sum',
 'sale_revenue_mean',
 'sale_discount_applied_sum',
 'is_returned_mean',
 'is_returned_last',
 'pack_delay_days_max',
 'is_jan_july_mean',
 'prod_web_only_mean',
 'prod_size_nunique',
 'recency_days',
 'discount_affinity',
 'recent_spend_ratio',
 'spend_trajectory_ratio',
 'churn_risk_ratio']

I'm gonna try handling outliers, maybe this would help the mae

In [16]:
upper_cap = np.percentile(y_train, 99)

In [17]:
y_train_clipped = np.clip(y_train, a_min=0, a_max=upper_cap)

Also log transforming the numerical features

In [18]:
for col in X_train.columns:
    if any(keyword in col for keyword in ["sum", "revenue", "count"]):
        X_train[col] = np.log1p(np.clip(X_train[col], 0, None))
        X_valid[col] = np.log1p(np.clip(X_valid[col], 0, None))

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

In [20]:
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TweedieRegressor(max_iter=2000))
])

In [21]:
param_grid = {
    "model__power": np.arange(0, 2, 0.2),
    "model__alpha": [0.0, 0.001, 0.01, 0.1],
    "model__link": ["log"]
}

In [22]:
grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    verbose=0
)

In [23]:
print("Starting model fitting with hyperparameter tunning ...")
grid.fit(X_train, y_train_clipped)

Starting model fitting with hyperparameter tunning ...


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=2000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.0, 0.001, ...], 'model__link': ['log'], 'model__power': array([0. , 0....4, 1.6, 1.8])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and para

In [24]:
print("\nBest Parameters Found:")
for key, value in grid.best_params_.items():
    print(f"  {key}: {value}")


Best Parameters Found:
  model__alpha: 0.0
  model__link: log
  model__power: 0.0


In [25]:
best_model = grid.best_estimator_

In [26]:
y_pred = best_model.predict(X_valid)

In [27]:
mae = mean_absolute_error(y_valid, y_pred)

In [28]:
spearman_corr, p_value = spearmanr(y_valid, y_pred)

In [29]:
print("-" * 30)
print(f"Validation MAE: {mae:.2f}")
print(f"Validation Spearman Correlation: {spearman_corr:.4f}")
print("-" * 30)

------------------------------
Validation MAE: 77.28
Validation Spearman Correlation: 0.3699
------------------------------


Let's see what happens if I log-transform the target variable

In [30]:
model = TransformedTargetRegressor(
    regressor=pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

In [31]:
param_grid2 = {
    "regressor__model__power": np.arange(0, 2, 0.2),
    "regressor__model__alpha": [0.0, 0.01, 0.1, 1.0]
}

In [32]:
grid2 = GridSearchCV(
    estimator=model,
    param_grid=param_grid2,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    error_score="raise"
)

In [33]:
grid2.fit(X_train, y_train_clipped)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",TransformedTa...iter=2000))]))
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__model__alpha': [0.0, 0.01, ...], 'regressor__model__power': array([0. , 0....4, 1.6, 1.8])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and paramet

In [34]:
print("\nBest Parameters Found:")
for key, value in grid2.best_params_.items():
    print(f"  {key}: {value}")


Best Parameters Found:
  regressor__model__alpha: 0.01
  regressor__model__power: 0.2


In [35]:
best_model2 = grid2.best_estimator_

In [36]:
y_pred2 = best_model2.predict(X_valid)

In [37]:
mae2 = mean_absolute_error(y_valid, y_pred2)

In [38]:
spearman_corr2, p_value = spearmanr(y_valid, y_pred2)

In [39]:
print("-" * 30)
print(f"Validation MAE: {mae2:.2f}")
print(f"Validation Spearman Correlation: {spearman_corr2:.4f}")
print("-" * 30)

------------------------------
Validation MAE: 68.05
Validation Spearman Correlation: 0.3757
------------------------------
